In [1]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as sm

# 1. FUNKTION ZUR DATENEXTRAKTION (Universal für Eurostat JSON)
def eurostat_json_to_df(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    time_index = data['dimension']['time']['category']['index'] 
    time_labels = data['dimension']['time']['category']['label'] 
    values = data['value'] 
    rows = []
    for label_key, position in time_index.items():
        pos_str = str(position)
        if pos_str in values:
            rows.append({'datum_raw': time_labels[label_key], 'preis': values[pos_str]})
    df = pd.DataFrame(rows)
    df['datum'] = pd.to_datetime(df['datum_raw'].str.replace('M', '-'), format='%Y-%m')
    return df[['datum', 'preis']].sort_values('datum')

# 2. KOMBINATION DER DREI QUELLEN (API-Merge)
# Quelle A: Butter Ladenpreise (HICP)
df_butter = eurostat_json_to_df('eurostat_butter_cpi.json').rename(columns={'preis': 'butter_cpi'})
# Quelle B: Molkerei Erzeugerpreise (PPI)
df_ppi = eurostat_json_to_df('eurostat_ppi_dairy.json').rename(columns={'preis': 'dairy_ppi'})
# Quelle C: Allgemeiner Milchprodukte-Trend (HICP Dairy)
df_dairy_gen = eurostat_json_to_df('eurostat_cpi_dairy.json').rename(columns={'preis': 'dairy_general_cpi'})

# Zusammenführen aller drei Quellen in eine Master-Tabelle
df_master = pd.merge(df_butter, df_ppi, on='datum', how='inner')
df_master = pd.merge(df_master, df_dairy_gen, on='datum', how='inner')

# Filter auf den gewünschten Zeitraum 2020 - 2024
df_master = df_master[(df_master['datum'] >= '2020-01-01')].reset_index(drop=True)

# 3. STATISTISCHES MODELL (Regression)
# Wir erklären den Butterpreis durch: PPI + General Dairy Trend + Saison (Monat)
df_master['month'] = df_master['datum'].dt.month
X = df_master[['dairy_ppi', 'dairy_general_cpi']]
X = sm.add_constant(X)
# Saisonale Dummies hinzufügen
month_dummies = pd.get_dummies(df_master['month'], prefix='mon', drop_first=True).astype(float)
X = pd.concat([X, month_dummies], axis=1)

# Regression berechnen
model = sm.OLS(df_master['butter_cpi'], X).fit()
df_master['predicted'] = model.predict(X)
df_master['residuals'] = df_master['butter_cpi'] - df_master['predicted']

# 4. SIGNIFIKANZ-ANALYSE (Z-Score > 1.96 = 95% Konfidenz)
std_resid = df_master['residuals'].std()
df_master['z_score'] = df_master['residuals'] / std_resid
df_master['is_anomaly'] = df_master['z_score'].abs() > 1.96

anomalies = df_master[df_master['is_anomaly']]

# 5. AUSGABE DER ERGEBNISSE
print(f"--- ANALYSE DER PREIS-ANOMALIEN (3 APIs kombiniert) ---")
if not anomalies.empty:
    for _, row in anomalies.iterrows():
        typ = "überraschend HOCH" if row['z_score'] > 0 else "überraschend NIEDRIG"
        print(f"Monat: {row['datum'].strftime('%B %Y')} -> Preis war {typ}")
        print(f"      (Abweichung: {row['residuals']:.2f} Indexpunkte)")
else:
    print("Keine signifikanten Anomalien im gewählten Zeitraum gefunden.")

# 6. INTERAKTIVE VISUALISIERUNG (1 von 7 geforderten)
fig = px.scatter(df_master, x='datum', y='residuals', color='is_anomaly',
                 title='Identifizierte Anomalien: Abweichung vom fairen Marktwert (PPI & Trend bereinigt)',
                 labels={'residuals': 'Unerklärte Preisdifferenz', 'datum': 'Zeitachse'},
                 hover_data=['butter_cpi', 'dairy_ppi'])
fig.add_hline(y=1.96*std_resid, line_dash="dash", line_color="red", annotation_text="Signifikant Hoch")
fig.add_hline(y=-1.96*std_resid, line_dash="dash", line_color="red", annotation_text="Signifikant Niedrig")
fig.show()


--- ANALYSE DER PREIS-ANOMALIEN (3 APIs kombiniert) ---
Monat: January 2022 -> Preis war überraschend NIEDRIG
      (Abweichung: -14.71 Indexpunkte)
Monat: February 2022 -> Preis war überraschend NIEDRIG
      (Abweichung: -14.33 Indexpunkte)
Monat: March 2022 -> Preis war überraschend NIEDRIG
      (Abweichung: -16.57 Indexpunkte)
Monat: December 2022 -> Preis war überraschend HOCH
      (Abweichung: 16.00 Indexpunkte)
